<a href="https://colab.research.google.com/github/volodymyr-holovan/DTA2026/blob/main/ML/Classification_DecisionTreeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overfitting & underfitting. Cross-validation

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Щоб результати були однаковими щоразу (відтворюваність)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Бібліотеки готові ✅")

Бібліотеки готові ✅


In [10]:
m = 1200
tenure   = np.random.randint(1, 72, m)           # місяців з нами
monthly  = np.random.normal(70, 25, m).clip(15, 150)  # щомісячна оплата, $
support  = np.random.poisson(1.5, m)             # звернень у підтримку за рік
age      = np.random.randint(18, 75, m)          # вік клієнта

# Прихована логіка ризику відтоку (модель її не знає):
risk = (
    -0.05 * tenure        # довше з нами → менший ризик
    + 0.02 * monthly      # дорожчий тариф → трохи більший ризик
    + 0.45 * support      # багато звернень у підтримку → більший ризик
    - 0.01 * age          # старші клієнти трохи лояльніші
    + np.random.normal(0, 0.7, m)
)
prob = 1 / (1 + np.exp(-(risk - 0.5)))   # перетворюємо ризик на ймовірність 0..1
churn = (np.random.rand(m) < prob).astype(int)

df = pd.DataFrame({
    "tenure": tenure, "monthly": monthly.round(1),
    "support": support, "age": age, "churn": churn,
})

print("Частка клієнтів, що пішли:", f"{df['churn'].mean():.1%}")
df.head()

Частка клієнтів, що пішли: 39.3%


,tenure,monthly,support,age,churn
0,52,21.7,1,21,0
1,15,39.8,1,20,0
2,61,43.2,4,73,0
3,21,87.1,0,46,1
4,24,65.9,1,69,1


In [11]:
from sklearn.model_selection import train_test_split

X = df[['tenure', 'monthly', 'support', 'age']]  # features - ознаки
y = df['churn']  # target - ціль

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Навчальна вибірка:", X_train.shape[0], "клієнтів")
print("Тестова вибірка:", X_test.shape[0], "клієнтів")

Навчальна вибірка: 960 клієнтів
Тестова вибірка: 240 клієнтів


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

deep_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
deep_tree.fit(X_train, y_train)

acc_train = accuracy_score(y_train, deep_tree.predict(X_train))
acc_test = accuracy_score(y_test, deep_tree.predict(X_test))

print("ГЛИБОКЕ дерево без обмежень (перенавчання):")
print(f"     Train accuract = {acc_train:.2%} ← майже ідеально..")
print(f"     Test accuract = {acc_test:.2%} ← ..а на нових даних гірше!")
print(f"     Розрив = {(acc_train - acc_test):.2%} - ознака зубріння\n")


shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
shallow.fit(X_train, y_train)

acc_train = accuracy_score(y_train, shallow.predict(X_train))
acc_test = accuracy_score(y_test, shallow.predict(X_test))

print("ПРОСТЕ дерево (max_depth=3):")
print(f"     Train accuract = {acc_train:.2%}")
print(f"     Test accuract = {acc_test:.2%}")
print(f"     Розрив = {(acc_train - acc_test):.2%} - модель надійніша (чесніша)")

ГЛИБОКЕ дерево без обмежень (перенавчання):
     Train accuract = 100.00% ← майже ідеально..
     Test accuract = 62.92% ← ..а на нових даних гірше!
     Розрив = 37.08% - ознака зубріння

ПРОСТЕ дерево (max_depth=3):
     Train accuract = 72.29%
     Test accuract = 59.17%
     Розрив = 13.12% - модель надійніша (чесніша)


> ПРАВИЛО: якщо train набагато більший за test - виконується перенавчання!  
Лікування: зменшити кількість ознак (або обмежити глубину), збільшити набір даних

## Cross-validation

In [13]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE),
    X, y, cv=5, scoring="accuracy"
)

In [14]:
print(f"Accuracy на кожному з 5 розбиттів:\n{np.round(scores, 3)}")
print(f"Середнє = {scores.mean():.2%} +- {scores.std():.2%}")

Accuracy на кожному з 5 розбиттів:
[0.688 0.733 0.65  0.675 0.638]
Середнє = 67.67% +- 3.34%
